In [1]:
import pandas as pd
import sys, os
from pathlib import Path
from src.common.minio_client import download_df_parquet

In [2]:
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

## Dataset final

In [4]:
df_final = download_df_parquet(access_key, secret_key,"grupo5/final/year=2025/month=01/dataset_final.parquet")


In [5]:
df_final.stop_id.value_counts()

stop_id
D43S    17839
640S    17278
635S    17214
631S    17085
629S    16959
        ...  
M07N       14
S10S        4
S12S        4
J18S        2
S0LN        1
Name: count, Length: 1037, dtype: int64

In [6]:
(df_final[df_final.is_unscheduled == False].stop_id.value_counts() > 0).sum()

np.int64(899)

In [7]:
df_final['stop_id'].str[:3].value_counts()

stop_id
635    33636
631    33588
D43    33511
629    33486
626    33422
       ...  
H18       94
J18       63
S12        4
S10        4
S0L        1
Name: count, Length: 523, dtype: int64

In [8]:
df_final.columns

Index(['date', 'match_key', 'stop_id', 'route_id', 'direction',
       'delay_seconds', 'lagged_delay_1', 'lagged_delay_2',
       'route_rolling_delay', 'actual_headway_seconds', 'is_unscheduled',
       'hour_sin', 'hour_cos', 'dow', 'is_weekend', 'target_delay_10m',
       'target_delay_20m', 'target_delay_30m', 'target_delay_45m',
       'target_delay_60m', 'station_delay_10m', 'station_delay_20m',
       'station_delay_30m', 'target_delay_end', 'delta_delay_10m',
       'delta_delay_20m', 'delta_delay_30m', 'delta_delay_45m',
       'delta_delay_60m', 'delta_delay_end', 'stops_to_end', 'merge_time',
       'scheduled_time_to_end', 'temp_extreme', 'n_eventos_afectando',
       'tipo_referente', 'afecta_previo', 'afecta_durante', 'afecta_despues',
       'category', 'num_updates', 'timestamp_start',
       'seconds_since_last_alert', 'is_alert_just_published',
       'seconds_to_next_alert', 'alert_in_next_15m', 'alert_in_next_30m'],
      dtype='str')

## GTFS Scheduled

In [9]:
START_DATE = "2025-01-01"
END_DATE = "2025-03-31"

dates = pd.date_range(start=START_DATE, end=END_DATE).strftime("%Y-%m-%d").tolist()
dfs = []
for date in dates:
    try:
        df_gtfs = download_df_parquet(access_key, secret_key,f"grupo5/cleaned/gtfs_clean_scheduled/date={date}/gtfs_scheduled_{date}.parquet")
    except:
        print(f"Could not download data for date: {date}")
        continue
    dfs.append(df_gtfs)
df = pd.concat(dfs, ignore_index=True)

## Cálculo de Matrices de adyacencia

In [11]:
import numpy as np

# 1. Sort the data to ensure chronological order for every single train trip
df = df.sort_values(by=['trip_uid', 'scheduled_seconds']).reset_index(drop=True)

# 2. Shift the columns to find the "next" stop for each train
df['next_stop_id'] = df.groupby('trip_uid')['stop_id'].shift(-1)
df['next_scheduled_seconds'] = df.groupby('trip_uid')['scheduled_seconds'].shift(-1)

# 3. Drop the last stop of every trip (it has no "next" stop)
edges_df = df.dropna(subset=['next_stop_id']).copy()

# 4. Calculate the travel time in seconds between the two stops
edges_df['travel_time'] = edges_df['next_scheduled_seconds'] - edges_df['scheduled_seconds']

In [12]:
# Eliminate 0s and negative travel times
edges_df = edges_df[edges_df['travel_time'] > 0] 

# 5. Aggregate to find unique edges and their median travel time
# We group by source (stop_id) and destination (next_stop_id)
graph_df = edges_df.groupby(['stop_id', 'next_stop_id']).agg(
    median_travel_time=('travel_time', 'median'),
    trip_count=('trip_uid', 'count') # Optional: useful to filter out anomalies
).reset_index()

# Optional: Filter out accidental/rare edges (e.g., track reroutes that only happened once)
#graph_df = graph_df[graph_df['trip_count'] > 5]

In [13]:
# Get a list of all unique nodes (stops)
nodes = sorted(list(set(df['stop_id'].unique()) | set(df['next_stop_id'].dropna().unique())))
n_nodes = len(nodes)

# Create a mapping from stop_id to a matrix index (0 to N-1)
node_to_idx = {stop_id: idx for idx, stop_id in enumerate(nodes)}

# Initialize a matrix of zeros
A_unweighted = np.zeros((n_nodes, n_nodes), dtype=np.float32)

# Populate the matrix with 1s where edges exist
for _, row in graph_df.iterrows():
    i = node_to_idx[row['stop_id']]
    j = node_to_idx[row['next_stop_id']]
    A_unweighted[i, j] = 1.0

print("Unweighted Adjacency Matrix Shape:", A_unweighted.shape)

Unweighted Adjacency Matrix Shape: (899, 899)


In [14]:
# Initialize the weighted matrix
A_weighted = np.zeros((n_nodes, n_nodes), dtype=np.float32)

# Calculate sigma (standard deviation of our median travel times)
distances = graph_df['median_travel_time'].values
sigma = distances.std()

# Populate the weighted matrix using the Gaussian Kernel
for _, row in graph_df.iterrows():
    i = node_to_idx[row['stop_id']]
    j = node_to_idx[row['next_stop_id']]
    
    dist = row['median_travel_time']
    # Calculate weight: closer stations approach 1.0, further stations approach 0.0
    weight = np.exp(- (dist ** 2) / (sigma ** 2))
    
    A_weighted[i, j] = weight

print("Weighted Adjacency Matrix Shape:", A_weighted.shape)

Weighted Adjacency Matrix Shape: (899, 899)


In [15]:
A_unweighted.sum()

np.float32(1393.0)

In [18]:
# Extract the 1D array of the 899 scheduled stops
scheduled_nodes_array = df['stop_id'].unique()

# Convert to a sorted Python list
scheduled_nodes_list = sorted(scheduled_nodes_array.tolist())

## Tensor generation

In [19]:
def build_spatiotemporal_tensor(df: pd.DataFrame, scheduled_nodes: list[str], freq: str = '15min'):
    """
    Transforms a flat event DataFrame into a (Time, Nodes, Features) tensor.
    """
    # 1. Round timestamps to the desired time bin
    # Make sure to use the 'merge_time' column from your script
    df['time_bin'] = pd.to_datetime(df['merge_time']).dt.floor(freq)
    
    # 2. Filter only to the scheduled nodes (the 899 we discussed)
    df = df[df['stop_id'].isin(scheduled_nodes)].copy()
    
    # 3. Aggregate events occurring in the same 15-minute window for each station
    # We define different aggregation rules depending on the feature type
    agg_rules = {
        'delay_seconds': 'mean',          # Average delay of scheduled trains
        'is_unscheduled': 'sum',          # Count of unscheduled trains
        'temp_extreme': 'max',            # Weather context
        'n_eventos_afectando': 'max',     # Event context
        'route_rolling_delay': 'mean',    # Average rolling delay on the route
        'actual_headway_seconds': 'mean', # Average headway in seconds
        'target_delay_10m': 'mean'        # Target variable (do this for all targets)
    }
    
    # Group by time and node
    grouped_df = df.groupby(['time_bin', 'stop_id']).agg(agg_rules)
    
    # 4. Create the Complete Grid (The Cartesian Product)
    # This guarantees that if a station had 0 trains, it still gets a row
    all_times = pd.date_range(start=df['time_bin'].min(), end=df['time_bin'].max(), freq=freq)
    all_nodes = sorted(scheduled_nodes) # MUST be sorted to match your Adjacency Matrix
    
    full_index = pd.MultiIndex.from_product(
        [all_times, all_nodes], 
        names=['time_bin', 'stop_id']
    )
    
    # 5. Reindex to the full grid
    full_df = grouped_df.reindex(full_index).reset_index()
    
    # 6. Smart Imputation for Missing Nodes
    # Rule A: Missing trains mean 0 delay and 0 unscheduled volume
    full_df['delay_seconds'] = full_df['delay_seconds'].fillna(0)
    full_df['is_unscheduled'] = full_df['is_unscheduled'].fillna(0)
    full_df['route_rolling_delay'] = full_df['route_rolling_delay'].fillna(0)
    full_df['actual_headway_seconds'] = full_df['actual_headway_seconds'].fillna(0)
    
    # Rule B: Context features (weather, events) persist over time. 
    # We group by station and forward-fill the last known value.
    context_cols = ['temp_extreme', 'n_eventos_afectando']
    full_df[context_cols] = full_df.groupby('stop_id')[context_cols].ffill()
    
    # If the very first time steps are NaN, back-fill them
    full_df[context_cols] = full_df.groupby('stop_id')[context_cols].bfill()
    
    # 7. Reshape into the 3D Tensor
    # Sort by time first, then node, so the reshape aligns perfectly
    full_df = full_df.sort_values(['time_bin', 'stop_id'])
    
    # Select final features for X and Y
    feature_cols = ['delay_seconds', 'is_unscheduled', 'temp_extreme', 
                    'n_eventos_afectando', 'route_rolling_delay', 'actual_headway_seconds']
    target_cols = ['target_delay_10m'] # Add the rest of your targets here
    
    T = len(all_times)
    N = len(all_nodes)
    F = len(feature_cols)
    H = len(target_cols)
    
    # Extract raw numpy arrays and reshape
    X_array = full_df[feature_cols].values
    Y_array = full_df[target_cols].values
    
    X_tensor = X_array.reshape(T, N, F)
    Y_tensor = Y_array.reshape(T, N, H)
    
    return X_tensor, Y_tensor, all_times, all_nodes

In [21]:
# Usage:
X, Y, times, nodes = build_spatiotemporal_tensor(df_final, scheduled_nodes_list)
print("X Shape (Time, Nodes, Features):", X.shape)
print("Y Shape (Time, Nodes, Targets):", Y.shape)

X Shape (Time, Nodes, Features): (2976, 899, 6)
Y Shape (Time, Nodes, Targets): (2976, 899, 1)


## DCRNN Training

In [35]:
# Division cronologica (Train / Val / Test)
import numpy as np

# Aseguramos formato numpy por si X e Y vienen como listas o tensores
X_np = np.asarray(X)
Y_np = np.asarray(Y)

# Convertimos nans a 0s para evitar problemas
X_np = np.nan_to_num(X_np, nan=0.0)
Y_np = np.nan_to_num(Y_np, nan=0.0)

T = X_np.shape[0]
train_end = int(T * 0.70)
val_end = train_end + int(T * 0.10)

X_train, X_val, X_test = X_np[:train_end], X_np[train_end:val_end], X_np[val_end:]
Y_train, Y_val, Y_test = Y_np[:train_end], Y_np[train_end:val_end], Y_np[val_end:]

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"Y_val shape:   {Y_val.shape}")
print(f"Y_test shape:  {Y_test.shape}")

X_train shape: (2083, 899, 6)
X_val shape:   (297, 899, 6)
X_test shape:  (596, 899, 6)
Y_train shape: (2083, 899, 1)
Y_val shape:   (297, 899, 1)
Y_test shape:  (596, 899, 1)


In [36]:
# Normalizacion de caracteristicas (Feature Scaling)
from sklearn.preprocessing import StandardScaler

# Reorganizamos a 2D para escalar por feature: (T*N, F)
T_train, N, F = X_train.shape
X_train_2d = X_train.reshape(-1, F)
X_val_2d = X_val.reshape(-1, F)
X_test_2d = X_test.reshape(-1, F)

scaler = StandardScaler()
scaler.fit(X_train_2d)  # Fit solo con train

X_train_scaled = scaler.transform(X_train_2d).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val_2d).reshape(X_val.shape)
X_test_scaled = scaler.transform(X_test_2d).reshape(X_test.shape)

# Y se deja sin escalar por ahora
print("Escalado completado")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_val_scaled shape:   {X_val_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")

Escalado completado
X_train_scaled shape: (2083, 899, 6)
X_val_scaled shape:   (297, 899, 6)
X_test_scaled shape:  (596, 899, 6)


In [37]:
# Creacion de ventana deslizante (PyTorch Dataset)
import torch
from torch.utils.data import Dataset

class SubwayDataset(Dataset):
    def __init__(self, X, Y, history_len=4):
        self.X = torch.as_tensor(X.copy(), dtype=torch.float32)
        self.Y = torch.as_tensor(Y.copy(), dtype=torch.float32)
        self.history_len = history_len

        if self.X.shape[0] != self.Y.shape[0]:
            raise ValueError("X e Y deben tener el mismo tamano temporal (eje T).")
        if self.history_len < 1:
            raise ValueError("history_len debe ser >= 1.")
        if self.X.shape[0] <= self.history_len:
            raise ValueError("T debe ser mayor que history_len para generar ventanas.")

    def __len__(self):
        # Cada muestra usa [t, ..., t+history_len-1] para predecir t+history_len
        return self.X.shape[0] - self.history_len

    def __getitem__(self, idx):
        x_window = self.X[idx : idx + self.history_len]          # (history_len, N, F)
        y_window = self.Y[idx + self.history_len].unsqueeze(0)   # (1, N, H)
        return x_window, y_window

In [38]:
# Generacion de DataLoaders
from torch.utils.data import DataLoader

history_len = 4
batch_size = 32

train_dataset = SubwayDataset(X_train_scaled, Y_train, history_len=history_len)
val_dataset = SubwayDataset(X_val_scaled, Y_val, history_len=history_len)
test_dataset = SubwayDataset(X_test_scaled, Y_test, history_len=history_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Batch de prueba para validar dimensiones
x_batch, y_batch = next(iter(train_loader))
print(f"x_batch shape: {x_batch.shape}  # (Batch, History, Nodes, Features)")
print(f"y_batch shape: {y_batch.shape}  # (Batch, 1, Nodes, Horizons)")

x_batch shape: torch.Size([32, 4, 899, 6])  # (Batch, History, Nodes, Features)
y_batch shape: torch.Size([32, 1, 899, 1])  # (Batch, 1, Nodes, Horizons)


In [45]:
# Preparacion del Grafo (PyTorch Geometric Format)
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
A_tensor = torch.as_tensor(A_weighted, dtype=torch.float32)

# Convertimos la matriz densa a representacion COO: edge_index [2, E], edge_weight [E]
edge_index = torch.nonzero(A_tensor != 0, as_tuple=False).t().contiguous()
edge_weight = A_tensor[edge_index[0], edge_index[1]].contiguous()

from torch_geometric.utils import add_self_loops

# Esto añade conexiones de cada nodo consigo mismo con un peso de 1.0 -> evitamos nans
edge_index, edge_weight = add_self_loops(
    edge_index, 
    edge_weight, 
    fill_value=1.0, 
    num_nodes=899
)

edge_index = edge_index.to(device)
edge_weight = edge_weight.to(device)

print(f"Device: {device}")
print(f"edge_index shape: {edge_index.shape}")
print(f"edge_weight shape: {edge_weight.shape}")
print(f"Numero de aristas: {edge_weight.numel()}")

Device: cpu
edge_index shape: torch.Size([2, 2292])
edge_weight shape: torch.Size([2292])
Numero de aristas: 2292


In [46]:
# Definicion de la Arquitectura del Modelo
import torch.nn as nn
from torch_geometric_temporal.nn.recurrent import DCRNN

class SubwayDCRNN(nn.Module):
    def __init__(self, in_channels=6, hidden_channels=32, out_horizons=1, K=2):
        super().__init__()
        self.dcrnn = DCRNN(in_channels=in_channels, out_channels=hidden_channels, K=K)
        self.readout = nn.Linear(hidden_channels, out_horizons)

    def forward(self, X, edge_index, edge_weight):
        # X: (Batch, History, Nodes, Features)
        B, H, N, F = X.shape
        hidden_state = None

        # Recorremos el eje temporal History y propagamos el estado oculto H
        for t in range(H):
            x_t = X[:, t, :, :]  # (Batch, Nodes, Features)
            next_hidden = []

            # DCRNN opera por grafo; procesamos cada elemento del batch
            for b in range(B):
                h_prev_b = None if hidden_state is None else hidden_state[b]
                h_b = self.dcrnn(x_t[b], edge_index, edge_weight, h_prev_b)  # (Nodes, hidden_channels)
                next_hidden.append(h_b)

            hidden_state = torch.stack(next_hidden, dim=0)  # (Batch, Nodes, hidden_channels)

        # Prediccion del siguiente paso temporal
        y_hat = self.readout(hidden_state)  # (Batch, Nodes, 1)
        y_hat = y_hat.unsqueeze(1)          # (Batch, 1, Nodes, 1)
        return y_hat

In [47]:
# Instanciacion y Configuracion
model = SubwayDCRNN(in_channels=6, hidden_channels=32, out_horizons=1, K=2).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4) 
criterion = torch.nn.L1Loss()  # MAE

print(model)
print(f"Modelo en device: {next(model.parameters()).device}")
print(f"Optimizador: Adam | lr=0.001")
print("Loss: L1Loss (MAE)")

SubwayDCRNN(
  (dcrnn): DCRNN(
    (conv_x_z): DConv(38, 32)
    (conv_x_r): DConv(38, 32)
    (conv_x_h): DConv(38, 32)
  )
  (readout): Linear(in_features=32, out_features=1, bias=True)
)
Modelo en device: cpu
Optimizador: Adam | lr=0.001
Loss: L1Loss (MAE)


In [48]:
print("¿Hay NaNs en X?", torch.isnan(torch.tensor(X_train)).any().item())
print("¿Hay NaNs en Y?", torch.isnan(torch.tensor(Y_train)).any().item())
print("¿Hay NaNs en los pesos del grafo?", torch.isnan(edge_weight).any().item())

¿Hay NaNs en X? False
¿Hay NaNs en Y? False
¿Hay NaNs en los pesos del grafo? False


In [50]:
# Bucle de Entrenamiento (Training & Validation Loop)
num_epochs = 10

for epoch in range(1, num_epochs + 1):
    model.train()
    train_running_loss = 0.0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        y_pred = model(x_batch, edge_index, edge_weight)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        
        # Por muy grande que sea el error, la actualización de los pesos nunca supere un límite
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        train_running_loss += loss.item()

    train_loss = train_running_loss / max(len(train_loader), 1)

    model.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            y_pred = model(x_batch, edge_index, edge_weight)
            val_loss_batch = criterion(y_pred, y_batch)
            val_running_loss += val_loss_batch.item()

    val_loss = val_running_loss / max(len(val_loader), 1)

    print(f"Epoch {epoch:02d}/{num_epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

Epoch 01/10 | Train Loss: 138.676473 | Val Loss: 117.698564
Epoch 02/10 | Train Loss: 138.627687 | Val Loss: 117.645596
Epoch 03/10 | Train Loss: 138.492013 | Val Loss: 117.526065
Epoch 04/10 | Train Loss: 138.327438 | Val Loss: 117.427353
Epoch 05/10 | Train Loss: 138.228533 | Val Loss: 117.356715
Epoch 06/10 | Train Loss: 138.166542 | Val Loss: 117.307305
Epoch 07/10 | Train Loss: 138.113828 | Val Loss: 117.260223
Epoch 08/10 | Train Loss: 138.067241 | Val Loss: 117.214497
Epoch 09/10 | Train Loss: 138.011798 | Val Loss: 117.167472
Epoch 10/10 | Train Loss: 137.963008 | Val Loss: 117.119572
